In [41]:
import zipfile
#inspect contents of zipfile
with zipfile.ZipFile('power_data.zip', 'r') as zip_ref:
    print(zip_ref.namelist())

['DOE_data_dictionary.csv', 'DOE_Electric_Disturbance_Events.xlsx', 'OE417_E-Filing_System_Training.pdf', 'OE417_Form_Instructions.pdf', 'OE417_Survey_Form.pdf']


In [42]:
import pandas as pd
import numpy as np
import plotly.express as px
from datetime import time
from dateutil.parser import parse
# Set pandas option to display all columns and rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)     # Show all rows


read in all the 21 excell sheets

In [43]:
#read in all excell sheets into a dictionary
def get_sheet_names():

    """read in excel sheets and returns a dictionary"""
    csv_file= 'DOE_Electric_Disturbance_Events.xlsx'
    zip_path = 'power_data.zip'
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        with zip_ref.open(csv_file) as file:
            sheets = pd.read_excel(file, sheet_name=None,skiprows=1)
    return sheets


In [44]:
sheets =get_sheet_names()
print(type(sheets))
# iterate through the disctionary
for k, v in sheets.items():
    print(k, type(v)) 

<class 'dict'>
2002 <class 'pandas.core.frame.DataFrame'>
2003 <class 'pandas.core.frame.DataFrame'>
2004 <class 'pandas.core.frame.DataFrame'>
2005 <class 'pandas.core.frame.DataFrame'>
2006 <class 'pandas.core.frame.DataFrame'>
2007 <class 'pandas.core.frame.DataFrame'>
2008 <class 'pandas.core.frame.DataFrame'>
2009 <class 'pandas.core.frame.DataFrame'>
2010 <class 'pandas.core.frame.DataFrame'>
2011 <class 'pandas.core.frame.DataFrame'>
2012 <class 'pandas.core.frame.DataFrame'>
2013 <class 'pandas.core.frame.DataFrame'>
2014 <class 'pandas.core.frame.DataFrame'>
2015 <class 'pandas.core.frame.DataFrame'>
2016 <class 'pandas.core.frame.DataFrame'>
2017 <class 'pandas.core.frame.DataFrame'>
2018 <class 'pandas.core.frame.DataFrame'>
2019 <class 'pandas.core.frame.DataFrame'>
2020 <class 'pandas.core.frame.DataFrame'>
2021 <class 'pandas.core.frame.DataFrame'>
2022 <class 'pandas.core.frame.DataFrame'>
2023 <class 'pandas.core.frame.DataFrame'>


In [45]:
list[sheets.keys()]

list[dict_keys(['2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023'])]

In [46]:
def extract_files(sheet_name):
    """ extract the excel sheets from the excel zip file
    parameters
    sheet_index"""
    csv_file= 'DOE_Electric_Disturbance_Events.xlsx'
    zip_path = 'power_data.zip'
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        with zip_ref.open(csv_file) as file:
            # Handle sheets 0 and 6 separately
            if sheet_name =='2002':
                df = pd.read_excel(file, sheet_name=sheet_name, skiprows=2)  # nrows=2 for sheet 0
                
            elif sheet_name== '2008':
                df = pd.read_excel(file, sheet_name=sheet_name, skiprows=2)  # nrows=2 for sheet 6
            else:
                df = pd.read_excel(file, sheet_name=sheet_name, skiprows=1)  # nrows=1 for other sheets
            #add a year column for each sheet
            try:
                year = int(sheet_name)
                df['year'] = year
            except ValueError:
                pass
              

    # Clean column names: convert to lowercase and replace spaces with hyphens
    df.columns = df.columns.str.lower().str.replace(" ", "-")
    # Drop rows where only one column has a value
    df = df.dropna(thresh = len(df.columns)-1)

    # function to handle column names
    def clean_column_names(df):

        #remove trailing spaces and characters
        df.columns = df.columns.str.strip()
        # Consolidate similar columns (e.g., 'area' and 'area-affected')
        df.rename(columns={
            'area': 'area-affected',
            'number-of-customers-affected-1': 'number-of-customers-affected',
            'number-of-customers-affected-1[1]': 'number-of-customers-affected',
            '-nerc-region': 'nerc-region',
            'date': 'date-event-began',
            'time': 'time-event-began',
            'date-of-restoration': 'restoration-date',
            'time-of-restoration': 'restoration-time',
            'event-type': 'type-of-disturbance',
            'demand-loss-(mw)': 'loss-(megawatts)',
                }, inplace=True)
        
                
        #drop unnecessary rows
        columns_to_drop = ['month', 'event-year', 'event-month',]
        df.drop(columns=[col for col in columns_to_drop if col in df.columns], inplace=True)

    # Call the clean_column_names function to modify column names
    clean_column_names(df)
        
 
    return df

In [47]:
#merge all sheets
dfs =[]

for year in range(2002, 2024):
    sheet_name = str(year)
    df = extract_files(sheet_name)
    dfs.append(df)
# merge all df into one
merged_df = pd.concat(dfs, ignore_index=True)
print(merged_df.shape)
merged_df.head()


(3814, 12)


,date-event-began,nerc-region,time-event-began,area-affected,type-of-disturbance,loss-(megawatts),number-of-customers-affected,restoration-time,year,restoration,restoration-date,alert-criteria
0,2002-01-30 00:00:00,SPP,06:00:00,Oklahoma,Ice Storm,500,1881134,2002-02-07 12:00:00,2002,NaN,NaN,NaN
1,2002-01-29 00:00:00,SPP,Evening,Metropolitan Kansas City Area,Ice Storm,500-600,270000,NaN,2002,NaN,NaN,NaN
2,2002-01-30 00:00:00,SPP,16:00:00,Missouri,Ice Storm,210,95000,2002-02-10 21:00:00,2002,NaN,NaN,NaN
3,2002-02-27 00:00:00,WSCC,10:48:00,California,Interruption of Firm Load,300,255000,2002-02-27 11:35:00,2002,NaN,NaN,NaN
4,2002-03-09 00:00:00,ECAR,00:00:00,Lower Peninsula of Michigan,Severe Weather,190,190000,2002-03-11 12:00:00,2002,NaN,NaN,NaN


In [48]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3814 entries, 0 to 3813
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   date-event-began              3814 non-null   object
 1   nerc-region                   3794 non-null   object
 2   time-event-began              3813 non-null   object
 3   area-affected                 3814 non-null   object
 4   type-of-disturbance           3814 non-null   object
 5   loss-(megawatts)              3593 non-null   object
 6   number-of-customers-affected  3771 non-null   object
 7   restoration-time              3057 non-null   object
 8   year                          3814 non-null   int64 
 9   restoration                   756 non-null    object
 10  restoration-date              3035 non-null   object
 11  alert-criteria                2259 non-null   object
dtypes: int64(1), object(11)
memory usage: 357.7+ KB


Handle object columns

In [49]:
def clean_object_columns(merged_df):

    #remove  rows with metadata
    rows_to_drop = [732,760,392,412]
    merged_df.drop(index=[idx for idx in rows_to_drop if idx in merged_df.index], inplace=True)
    # handle inconsistencies in the nerc-region
    def clean_nerc_region(region):
        #handle nan values
        if pd.isna(region):
            return np.nan
        # remove trailing spaces and capitalise
        region = (region.strip().upper().replace(" / ", "/")
                  .replace(" ,", ",").replace(", ", ",").replace(",", "/").replace(";", ",")
                  .replace(", ","/")
                  )
        #fix common typos
        replacements = {'MIDWEST ISO (RFC':'MIDWEST ISO (RFC)',
                        'MR0':'MRO','NPPC':'NPCC','SPP RE':'SPP/RE','NERC REGION':'NERC/REGION'}
        region = replacements.get(region, region)  # Apply replacements
        return region
    # Apply cleaning function
    merged_df["nerc-region"] = merged_df["nerc-region"].astype(str).apply(clean_nerc_region)

    return merged_df

merged_df  = clean_object_columns(merged_df)    

handle area affected column. got the uscities dataset from https://simplemaps.com/data/us-cities

In [50]:
import json
# Load city-to-state mapping
with open("city_to_state.json", "r") as f:
    CITY_TO_STATE = json.load(f)
# Display the first key-value pairs
first_items = list(CITY_TO_STATE.items())[:2]  
print(first_items)

# Load county-to-state mapping
with open("county_to_state.json", "r") as f:
    COUNTY_TO_STATE = json.load(f)
# Display the first key-value pairs
print(list(COUNTY_TO_STATE.items())[:5] )

#load state names
STATE_NAMES = set(CITY_TO_STATE.values())
print('Arizona' in STATE_NAMES)

[('New York', 'New York'), ('Los Angeles', 'Texas')]
[('Queens', 'New York'), ('Los Angeles', 'California'), ('Cook', 'Minnesota'), ('Miami-Dade', 'Florida'), ('Harris', 'Texas')]
True


In [51]:
STATE_NAMES = set(CITY_TO_STATE.values())
print('Puerto Rico' in STATE_NAMES)

True


In [52]:
import re
import pandas as pd
from fuzzywuzzy import process

def clean_area(area, threshold = 65):
    """Clean the area to extract a valid state."""

    # Return NaN for invalid or empty input
    invalid_list = ["-", " ", "", "Unknown", "Entergy System", "ISO Balancing", "Restricted Hydroelectric Capability", 
                    "Coastal areas of", "Company Territory", "Location Unknown", "Balancing", "ISO Balancing"]

    # Clean the area by stripping whitespace and checking for invalid entries
    cleaned_area = re.sub(r'\s+', ' ', area).strip()  # Replace multiple spaces with a single space
    
    if not isinstance(cleaned_area, str) or cleaned_area in invalid_list:
        return pd.NA
    #Apply replacement for known abbreviations 
    replacements = {
        "NJ": "New Jersey",
        "NY": "New York",
        "and  Kentrucky": "Kentucky",
        "of Los Angeles": "Los Angeles",
        "San Antonio, TX": "Texas",
        "Houston-Galveston Metro": "Texas",
        "San Diego": "California",
        "The Entire ComEd Territory": "Illinois",
        "The Entire ComEd Service Territory": "Illinois",
        "The ComEd Territory": "Illinois",
        "Entire BGE Service Territory": "Maryland",
        "Dominion Service Territory": "Virginia",
        "TVA Service Territory": "Tennessee",
        "Puget Sound Region": "Washington",
        "Part of Seattle's Downtown": "Washington",
        "Los Angeles": "California"
        }
    
    # Loop through the replacements dictionary and apply each replacement
    for old, new in replacements.items():
        area = area.replace(old, new)
    
    # Remove unwanted keywords like  'Area', 'Region', etc.
    area = re.sub(
        r'\b(Central|Eastern|Southern|Western|North/Central|Southeastern|Northeast|Southeast|County|City|Area|Greater)\b', '', area).strip()

    # Split the string by common delimiters
    locations = re.split(r'[ ,\;&:/]+|\band\b|\bof\b', area)

    # Check for direct state matches first
    state_matches = [
        CITY_TO_STATE[state] for state in CITY_TO_STATE.keys() if state in locations
    ]
    
    if state_matches:
        return state_matches[0]  # Return the first match
    
    # Check for city matches
    for city in CITY_TO_STATE.keys():
        if city in locations:
            return CITY_TO_STATE[city]  # Return corresponding state

    # Check for county matches (handling multiple counties)
    county_states = [
        COUNTY_TO_STATE[county] for county in COUNTY_TO_STATE.keys() if county in locations
    ]
    
    if county_states:
        # Return the most frequent state among matched counties
        return max(set(county_states), key=county_states.count) 
    

    # Fuzzy matching as a last resort
    match, score = process.extractOne(area, STATE_NAMES)
    return match if score >= threshold else pd.NA
    

# Apply the function to the DataFrame
merged_df['area-affected'] = merged_df['area-affected'].apply(clean_area)



handle loss-(megawatts) column

In [53]:
def handle_ranges(x):
    # Handle ranges and invalid entries
    try:
        if isinstance(x, str):
            # Check if it's just a '-' (or other invalid case)
            if x.strip() == '-' or x.strip() == '':
                return pd.NA
            
            # Handle ranges like '500-600'
            if '-' in x:
                parts = x.split('-')
                return (float(parts[0]) + float(parts[1])) / 2
            # Handle ranges like '80 to 100'
            if 'to' in x:
                parts = x.replace(',','').split('to')
                return (float(parts[0]) + float(parts[1])) / 2
            # Handle ranges like '37- 40'
            if '- ' in x:
                parts = x.split('- ')
                return (float(parts[0]) + float(parts[1])) / 2     
        
        # If it's not a range, just return the string as-is for further cleaning
        return x

    except:
        # In case of an error, return the original value
        return x
# call function
merged_df['loss-(megawatts)'] = merged_df['loss-(megawatts)'].astype(str).apply(handle_ranges)

In [54]:
def process_value(x):
    try:
        if isinstance(x, str):
            # Handle missing/invalid cases
            invalid = ['-', '--', 'N/A', 'unknown', 'All', 'Loss (megawatts)', 'nan', 'Unknown', 'UNK']
            if pd.isna(x) or x.strip() in invalid:
                return pd.NA
            
            # Extract numbers at the beginning of a string
            num_match = re.search(r"^\d+", x)
            if num_match:
                return float(num_match.group())  
            
            # Extract numbers at the end of the string
            match = re.search(r"\d+(\.\d+)?\s*$", x)
            if match:
                return float(match.group())  
            
            # Extract numbers appearing anywhere in the string (handles comma separators)
            num_str = re.search(r"\d{1,3}(,\d{3})*(\.\d+)?", x)
            if num_str:
                return float(num_str.group().replace(",", ""))  # Remove commas and convert to float
            
        return pd.NA  # If no match, return nan
    except:
        return pd.NA

# Apply function to column
merged_df['loss-(megawatts)'] = merged_df['loss-(megawatts)'].astype(str).apply(process_value)
merged_df['number-of-customers-affected'] = merged_df['number-of-customers-affected'].astype(str).apply(process_value)
           

handle type-of-disturbance column. it was grouped to reduce cardinality

In [55]:
def clean_disturbance_type(merged_df):
    # Remove trailing spaces
    merged_df['type-of-disturbance'] = merged_df['type-of-disturbance'].str.strip()
    # Standardize case
    merged_df['type-of-disturbance'] = merged_df['type-of-disturbance'].str.lower()
    # Replace '- Unknown' with 'Unknown'
    merged_df['type-of-disturbance'] = merged_df['type-of-disturbance'].replace('- unknown', 'unknown')
    # Remove content in parentheses
    merged_df['type-of-disturbance'] = merged_df['type-of-disturbance'].str.replace(r'\(.*\)', '', regex=True)
    #replace hyphen with slashes
    merged_df['type-of-disturbance'] = merged_df['type-of-disturbance'].str.replace(' - ', '/', regex=True) 
    merged_df['type-of-disturbance'] = merged_df['type-of-disturbance'].str.replace('/', ',', regex=True) 
    #remove extra spaces
    merged_df['type-of-disturbance'] = merged_df['type-of-disturbance'].str.replace(r'\s+', ' ', regex=True)
    
    return merged_df

# Call function
merged_df = clean_disturbance_type(merged_df)


In [56]:
# Define regex patterns for each category
patterns = {
    'Weather Events': r'(weather|storm|thunderstorm|wind|hurricane|flood|fire|lightning|earthquake|tornado|ice)',  
    'Transmission Issues': r'(transmission|system|generation|shedding|voltage|power|load|interruption|transformer)',  
    'Security and Vandalism': r'(vandalism|sabotage|attack|physical|cut)',  
    'System & Fuel Issues': r'(fuel|supply|deficiency|tripped|fault|unit|failure|malfunction|loss|breaker|islanding)',  
    'Public Alerts': r'(public appeal|emergency|disaster)',  
    'Suspicious Activities and Cyber Events': r'(suspicious|cyber)',  
}

# Function to categorize events based on regex patterns
def categorize_event(event):
    
    for group, pattern in patterns.items():
        if re.search(pattern, event, re.IGNORECASE):  # Ignore case for matching
            return group
    return pd.NA

# Apply the function to the column
merged_df['type-of-disturbance'] = merged_df['type-of-disturbance'].apply(categorize_event)



handle alert criteria

In [57]:
def handle_alert_col(merged_df):
    # Remove trailing spaces
    merged_df['alert-criteria'] = merged_df['alert-criteria'].astype(str)
    merged_df['alert-criteria'] = merged_df['alert-criteria'].str.strip()
    merged_df['alert-criteria'] = merged_df['alert-criteria'].str.lower()
    #replace hyphen with slashes
    merged_df['alert-criteria'] = merged_df['alert-criteria'].str.replace(r"-\s\d+.\s",'', regex=True)
    merged_df['alert-criteria'] = merged_df['alert-criteria'].str.replace(r"\d.\s",'', regex=True) 
    return merged_df

# Call function
merged_df = handle_alert_col(merged_df)
    


In [58]:
# Apply the function to the alert criteria column

merged_df['alert-criteria'] = merged_df['alert-criteria'].apply(categorize_event)
#combine two similar columns
merged_df['cause']= merged_df['type-of-disturbance'].combine_first(merged_df['alert-criteria'])
#drop un necessary columns
merged_df.drop(columns=['alert-criteria','type-of-disturbance'], axis=1, inplace= True)


Handle all datetime columns

In [59]:

from datetime import time
from dateutil.parser import parse

def clean_time_cols(df):

    # Handle non-time entries by replacing 'Time' with NaT
    df['time-event-began'] = df['time-event-began'].apply(lambda x: pd.NaT if x == 'Time' else x)

    # Function to process time strings
    def convert_time(x):
        if isinstance(x, str):
            x = x.strip().lower()
            # Handle specific cases
            if x == 'evening':
                return time(15, 0)  # 3:00 PM
            if x in ['restoration', 'ongoing', 'restoration time', 'approximately']:
                return pd.NaT
            # Replace "noon" and "midnight"
            x = re.sub(r"\bnoon\b", "12:00 PM", x, flags=re.IGNORECASE)
            x = re.sub(r"\bmidnight\b", "12:00 AM", x, flags=re.IGNORECASE)

        # Convert to HH:MM:SS format if possible
        try:
            return parse(x).strftime("%H:%M:%S")
        except:
            return x

    # Apply function to relevant columns
    df['time-event-began'] = df['time-event-began'].apply(convert_time)
    df['restoration-time'] = df['restoration-time'].apply(convert_time)

    return df

# Calling the function
merged_df = clean_time_cols(merged_df)



In [60]:

def parse_date(date_str):
    """Convert various date formats to a consistent YYYY-MM-DD format"""
    
    # Handle invalid entries
    invalid_lst = ['date', 'date event began', 'unknown', 'ongoing', 'date of restoration']
    if not isinstance(date_str, str) or date_str.lower().strip() in  invalid_lst:
        return pd.NaT  # Return NaT for missing or invalid values
          
    # Convert to '%Y-%m-%d' format if possible using dateutil
    try:
        return parse(date_str).strftime('%Y-%m-%d')
    except:
        return date_str
  
# Apply function to convert dates in the DataFrame
merged_df['date-event-began'] = merged_df['date-event-began'].apply(parse_date)
merged_df['restoration-date'] = merged_df['restoration-date'].apply(parse_date)

   

In [61]:
def handle_time(time_str):
    """Convert various date/time formats to a consistent format"""
    
    # Handle invalid entries
    invalid = ['restoration','ongoing','restoration time','approximately']
    if not isinstance(time_str, str) or time_str.lower().strip() in  invalid:
        return pd.NaT  # Return NaT for missing or invalid values
    # Replace "noon" and "midnight"
    time_str= re.sub(r"\bnoon\b", "PM", time_str, flags=re.IGNORECASE)
    time_str = re.sub(r"\bmidnight\b", "AM", time_str, flags=re.IGNORECASE)
    
    # Convert using pandas datetime
    converted_time = pd.to_datetime(time_str, errors='coerce')
        
    # Return only the converted time if conversion is successful
    return converted_time if pd.notna(converted_time) else pd.NaT

# Apply function to convert dates in the DataFrame
merged_df['restoration'] = merged_df['restoration'].apply(handle_time)



In [62]:
#extract date & time  to be used in combining restoratime-time/restoration-date

# Extract date and time into separate columns
merged_df['date'] = merged_df['restoration'].dt.date
merged_df['time'] = merged_df['restoration'].dt.time

In [63]:
merged_df['time_event_restored'] = merged_df['restoration-time'].combine_first(merged_df['time'])
merged_df['date_event_restored'] = merged_df['restoration-date'].combine_first(merged_df['date'])

In [64]:
#drop un necessary columns
merged_df.drop(columns=['restoration','restoration-date','date','time'], axis=1, inplace= True)

In [65]:
merged_df.drop(columns=['restoration-time'], axis=1, inplace= True)

In [70]:
#combine date and time columns
merged_df['date_event_restored'] = merged_df['time_event_restored'].combine_first(merged_df['date_event_restored'])
merged_df['date-event-began'] = merged_df['time-event-began'].combine_first(merged_df['date-event-began'])

In [72]:
#save the clean df
merged_df.to_csv('power_df.csv', index= False)

In [71]:
(merged_df.isnull().sum()/3810)*100

date-event-began                 0.577428
nerc-region                      0.000000
time-event-began                 0.603675
area-affected                    1.286089
loss-(megawatts)                35.433071
number-of-customers-affected    15.931759
year                             0.000000
cause                            1.154856
time_event_restored              0.892388
date_event_restored              0.892388
dtype: float64

In [68]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3810 entries, 0 to 3813
Data columns (total 10 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   date-event-began              1966 non-null   object
 1   nerc-region                   3810 non-null   object
 2   time-event-began              3787 non-null   object
 3   area-affected                 3761 non-null   object
 4   loss-(megawatts)              2460 non-null   object
 5   number-of-customers-affected  3203 non-null   object
 6   year                          3810 non-null   int64 
 7   cause                         3766 non-null   object
 8   time_event_restored           3776 non-null   object
 9   date_event_restored           2395 non-null   object
dtypes: int64(1), object(9)
memory usage: 456.5+ KB
